# 正则化（超大白话版）

## 这份笔记是干嘛的？

把正则化这个听起来很唬人的名词，翻译成你能讲给别人听的版本：

- 为什么模型会过拟合
- 什么叫正则化
- 常见正则化方法到底在干什么
- 为什么它们能让模型别学太死
- 在 Keras 里怎么写

---
# 一、正则化到底想解决什么问题？

先记一句最重要的话：

> 正则化，就是想办法让模型别把训练集背得太死。

深度学习模型能力很强，参数很多。如果训练数据不够多，或者模型太大，就很容易出现：

- 训练集上效果越来越好
- 验证集/测试集上效果却变差

这就叫过拟合（Overfitting）。

大白话：模型不是在学规律，而是在记题库答案，训练集一换，它就不会了。

所以正则化的目标就是：让模型更克制一点，别乱记细节，尽量去学更通用的规律。

---
# 二、什么叫泛化能力？

> 模型对没见过的新数据，表现还行不行。

比如你训练时给它看了很多影评，测试时来了没见过的新影评，它还能判断对不对——这就是泛化能力。

如果只会做训练集里的题，换一批数据就崩，那就泛化差。

正则化，本质上就是为了提高泛化能力。

---
# 三、把模型想成学生（最好记的比喻）

| 概念 | 对应学生世界 |
|---|---|
| 训练集 | 平时练习题 |
| 验证集/测试集 | 考场新题 |
| 过拟合 | 死记硬背答案 |
| 正则化 | 不让他死记硬背，逼他理解题型 |

> 正则化不是让模型更猛，而是让模型更稳。

---
# 四、三种常见正则化方法（主线）

1. 减小网络规模
2. 权重正则化（L1 / L2）
3. Dropout

统一理解成一句话：

> 都是在给模型上约束，让它不要太自由。

因为模型越自由，越容易把训练集里那些没用的小噪声也学进去。

---
# 五、方法一：减小网络规模（最直观）

就是把模型做小一点——层数少一点，每层神经元少一点，参数少一点。

模型能力太强时，它可以把训练集记得特别细。把模型变小，记忆容量就下降了，它就只能优先学更重要的模式。

> 大白话：模型原来脑子太大，什么都硬记；现在脑容量限制一下，它只能记重点。

但模型太小会欠拟合，合适才最好。

In [ ]:
# 对比：偏大的模型 vs 更小的模型
from tensorflow import keras
from tensorflow.keras import layers

# 偏大的模型 - 容量大容易过拟合
model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

# 更小的模型 - 容量小更抗过拟合
small_model = keras.Sequential([
    layers.Dense(4, activation="relu"),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

---
# 六、方法二：权重正则化（重点）

神经网络里，输入和输出之间连线上的那些系数就是权重。

权重正则化在干嘛？不希望权重长得太大。

因为某些权重特别大的时候，模型会变得很激进，更容易过拟合。

所以我们给损失函数额外加一个惩罚项——权重越大，罚得越多。

---
# 七、L2 正则化：最常见、最常用

L2 正则化就是：如果权重太大，就罚你。罚的是权重平方和。

它逼模型：不要过度依赖某几个特别大的权重，尽量把信息分散地、平滑地表示出来。

> 类比：L2就像老师说：别耍小聪明，给我稳一点、均衡一点。

In [ ]:
# L2 正则化：在 Keras 里怎么写
from tensorflow.keras import regularizers

l2_model = keras.Sequential([
    layers.Dense(16, kernel_regularizer=regularizers.l2(0.001), activation="relu"),
    layers.Dense(16, kernel_regularizer=regularizers.l2(0.001), activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

# 0.001 是 lambda（正则化强度）
# 太小：几乎没效果
# 太大：模型被管得太狠，学不动

---
# 八、L1 正则化

L1 和 L2 很像，区别在惩罚方式：

| | 惩罚什么 | 效果 |
|---|---|---|
| L1 | 权重绝对值之和 | 更容易把权重压成0 |
| L2 | 权重平方和 | 把权重整体压小 |

L1更像做特征选择，L2更像让整体更平滑。

In [ ]:
# L1 正则化
l1_model = keras.Sequential([
    layers.Dense(16, kernel_regularizer=regularizers.l1(0.001), activation="relu"),
    layers.Dense(16, kernel_regularizer=regularizers.l1(0.001), activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

---
# 九、怎么看图判断正则化有没有用？

没加正则化时：训练loss不断下降，验证loss先下降后上升（过拟合）

加了正则化后：训练loss没那么漂亮，但验证loss更低或上升更慢（泛化好）

> 正则化不是为了让训练集成绩更好，而是为了让验证集/测试集表现更好。

---
# 十、方法三：Dropout（超级常用）

Dropout是什么？训练时随机把一部分神经元临时关掉。

为什么有用？模型就不能太依赖某几个固定神经元。

> 大白话：就像小组作业时，老师随机抽人缺席，结果全组成员都得会做，不能只靠一个学霸。

In [ ]:
# Dropout：在 Keras 里怎么写
dropout_model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.5),  # 随机关掉50%神经元
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

# Dropout比例选择：
# 0.2 丢得少，正则化轻
# 0.5 最常见
# 0.8 丢太多，学不动

---
# 十一、三种方法对比

| 方法 | 它在干什么 | 大白话效果 |
|---|---|---|
| 减小网络规模 | 参数变少 | 不容易死记硬背 |
| L1/L2正则化 | 给大权重加惩罚 | 让模型别太激进 |
| Dropout | 随机关掉神经元 | 不准只靠少数神经元 |

---
# 十二、容易混淆的问题

为什么加了正则化后，训练误差反而更差了？这是正常的！正则化本来就是给模型增加限制。

正则化是不是越强越好？不是。太强就变成欠拟合。

类比做菜放盐：不放太淡，放一点刚好，放太多齁咸。

---
# 十三、总逻辑

一句话版本：模型过拟合是因为对训练集学得太死；正则化通过各种方式限制模型，少记细节、多学规律，提升泛化能力。

展开版本：
1. 模型太强容易过拟合
2. 过拟合时训练集好但验证集差
3. 所以需要正则化
4. 正则化的本质是给模型加约束
5. 常见方法：减小网络规模、L1/L2、Dropout
6. 训练集没那么完美，但验证集更好

---
# 十四、最后一句话

> 正则化的本质，就是牺牲一点训练集上的爽感，换来测试集上的靠谱。

不让模型把答案背死，而是逼它学会做题的方法。